In [130]:
import numpy as np
import pandas as pd
import glob
import os


## 1 Data Cleaning

In [131]:
folder_path = "/Users/fuyanting/Documents/capstone/data"
files = glob.glob(os.path.join(folder_path, "*.csv"))

use_cols = [
    'ID',
    'Date',
    'Primary Type',
    'Description',
    'Location Description',
    'Arrest',
    'Domestic',
    'District',
    'Community Area',
    'Year',
    'Latitude',
    'Longitude'
]

df_list = []

for file in files:
    print(f"Reading: {file}") 
    
    temp = pd.read_csv(
        file,
        usecols=use_cols,
        low_memory=False
    )
    
    df_list.append(temp)


df = pd.concat(df_list, ignore_index=True)
print("\nFinal shape:", df.shape)
print("\nYear distribution:\n", df['Year'].value_counts().sort_index())

Reading: /Users/fuyanting/Documents/capstone/data/Crimes_2016.csv
Reading: /Users/fuyanting/Documents/capstone/data/Crimes_2017.csv
Reading: /Users/fuyanting/Documents/capstone/data/Crimes_2020.csv
Reading: /Users/fuyanting/Documents/capstone/data/Crimes_2021.csv
Reading: /Users/fuyanting/Documents/capstone/data/Crimes_2023.csv
Reading: /Users/fuyanting/Documents/capstone/data/Crimes_2022.csv
Reading: /Users/fuyanting/Documents/capstone/data/Crimes_2025.csv
Reading: /Users/fuyanting/Documents/capstone/data/Crimes_2019.csv
Reading: /Users/fuyanting/Documents/capstone/data/Crimes_2018.csv
Reading: /Users/fuyanting/Documents/capstone/data/Crimes_2024.csv

Final shape: (2491875, 12)

Year distribution:
 Year
2016    269968
2017    269292
2018    269155
2019    261709
2020    212720
2021    209679
2022    240034
2023    263311
2024    259073
2025    236934
Name: count, dtype: int64


In [132]:
df.head()

,ID,Date,Primary Type,Description,Location Description,Arrest,Domestic,District,Community Area,Year,Latitude,Longitude
0,13211439,01/01/2016 12:00:00 AM,OFFENSE INVOLVING CHILDREN,AGGRAVATED SEXUAL ASSAULT OF CHILD BY FAMILY M...,RESIDENCE,False,True,8.0,63.0,2016,NaN,NaN
1,13047536,01/01/2016 12:00:00 AM,OFFENSE INVOLVING CHILDREN,AGGRAVATED SEXUAL ASSAULT OF CHILD BY FAMILY M...,APARTMENT,True,True,10.0,30.0,2016,41.837918,-87.713393
2,12328317,01/01/2016 12:00:00 AM,SEX OFFENSE,CRIMINAL SEXUAL ABUSE,APARTMENT,False,False,10.0,30.0,2016,41.849026,-87.708830
3,12389557,01/01/2016 12:00:00 AM,SEX OFFENSE,SEXUAL EXPLOITATION OF A CHILD,RESIDENCE,True,True,4.0,43.0,2016,NaN,NaN
4,13278509,01/01/2016 12:00:00 AM,SEX OFFENSE,AGGRAVATED CRIMINAL SEXUAL ABUSE,RESIDENCE,False,True,8.0,58.0,2016,NaN,NaN


In [133]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2491875 entries, 0 to 2491874
Data columns (total 12 columns):
 #   Column                Dtype  
---  ------                -----  
 0   ID                    int64  
 1   Date                  object 
 2   Primary Type          object 
 3   Description           object 
 4   Location Description  object 
 5   Arrest                bool   
 6   Domestic              bool   
 7   District              float64
 8   Community Area        float64
 9   Year                  int64  
 10  Latitude              float64
 11  Longitude             float64
dtypes: bool(2), float64(4), int64(2), object(4)
memory usage: 194.9+ MB


In [134]:
print(df.isna().sum())
print('-----------')
print((df.isna().mean() * 100).round(2))

ID                          0
Date                        0
Primary Type                0
Description                 0
Location Description    13168
Arrest                      0
Domestic                    0
District                    1
Community Area            206
Year                        0
Latitude                36957
Longitude               36957
dtype: int64
-----------
ID                      0.00
Date                    0.00
Primary Type            0.00
Description             0.00
Location Description    0.53
Arrest                  0.00
Domestic                0.00
District                0.00
Community Area          0.01
Year                    0.00
Latitude                1.48
Longitude               1.48
dtype: float64


In [135]:
df['Location Description'] = df['Location Description'].fillna('Unknown')
df = df.dropna(subset=['Community Area','District'])
df_map = df.dropna(subset=['Latitude', 'Longitude'])

In [136]:
df['ID'] = pd.to_numeric(df['ID'], errors='coerce').astype('Int32')

df['District'] = pd.to_numeric(df['District'], errors='coerce').astype('Int16')
df['Community Area'] = pd.to_numeric(df['Community Area'], errors='coerce').astype('Int16')
df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int16')

df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce').astype('float32')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce').astype('float32')

df['Arrest'] = df['Arrest'].astype('boolean')
df['Domestic'] = df['Domestic'].astype('boolean')

cat_cols = ['Primary Type', 'Description', 'Location Description']
for col in cat_cols:
    df[col] = df[col].astype('category')

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

/var/folders/y4/3yldd68d36qfw_vh1kk8kp5m0000gn/T/ipykernel_41845/1233118722.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'], errors='coerce')


In [137]:
print(df.isna().sum())

ID                          0
Date                        0
Primary Type                0
Description                 0
Location Description        0
Arrest                      0
Domestic                    0
District                    0
Community Area              0
Year                        0
Latitude                36953
Longitude               36953
dtype: int64


In [138]:
df.duplicated().sum()

0

In [139]:
df['Month'] = df['Date'].dt.month.astype('int8')
df['Hour'] = df['Date'].dt.hour.astype('int8')
df['DayOfWeek'] = df['Date'].dt.dayofweek.astype('int8')

df_map[['Month','Hour','DayOfWeek']] = df[['Month','Hour','DayOfWeek']]

/var/folders/y4/3yldd68d36qfw_vh1kk8kp5m0000gn/T/ipykernel_41845/3761630759.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_map[['Month','Hour','DayOfWeek']] = df[['Month','Hour','DayOfWeek']]
/var/folders/y4/3yldd68d36qfw_vh1kk8kp5m0000gn/T/ipykernel_41845/3761630759.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_map[['Month','Hour','DayOfWeek']] = df[['Month','Hour','DayOfWeek']]
/var/folders/y4/3yldd68d36qfw_vh1kk8kp5m0000gn/T/ipykernel_41845/3761630759.py:5: SettingWithCopyWarning: 
A val

In [140]:
df.head()

,ID,Date,Primary Type,Description,Location Description,Arrest,Domestic,District,Community Area,Year,Latitude,Longitude,Month,Hour,DayOfWeek
0,13211439,2016-01-01,OFFENSE INVOLVING CHILDREN,AGGRAVATED SEXUAL ASSAULT OF CHILD BY FAMILY M...,RESIDENCE,False,True,8,63,2016,NaN,NaN,1,0,4
1,13047536,2016-01-01,OFFENSE INVOLVING CHILDREN,AGGRAVATED SEXUAL ASSAULT OF CHILD BY FAMILY M...,APARTMENT,True,True,10,30,2016,41.837917,-87.713394,1,0,4
2,12328317,2016-01-01,SEX OFFENSE,CRIMINAL SEXUAL ABUSE,APARTMENT,False,False,10,30,2016,41.849026,-87.708832,1,0,4
3,12389557,2016-01-01,SEX OFFENSE,SEXUAL EXPLOITATION OF A CHILD,RESIDENCE,True,True,4,43,2016,NaN,NaN,1,0,4
4,13278509,2016-01-01,SEX OFFENSE,AGGRAVATED CRIMINAL SEXUAL ABUSE,RESIDENCE,False,True,8,58,2016,NaN,NaN,1,0,4


In [141]:
df.describe()

,ID,Date,District,Community Area,Year,Latitude,Longitude,Month,Hour,DayOfWeek
count,2491668.0,2491668,2491668.0,2491668.0,2491668.0,2.454715e+06,2.454715e+06,2.491668e+06,2.491668e+06,2.491668e+06
mean,12197439.472684,2020-11-30 01:57:23.599163648,11.253122,36.599791,2020.40648,4.184455e+01,-8.766965e+01,6.610433e+00,1.274579e+01,3.012063e+00
min,22245.0,2016-01-01 00:00:00,1.0,1.0,2016.0,3.661945e+01,-9.168657e+01,1.000000e+00,0.000000e+00,0.000000e+00
25%,11308602.75,2018-05-04 20:29:45,6.0,23.0,2018.0,4.176926e+01,-8.771163e+01,4.000000e+00,8.000000e+00,1.000000e+00
50%,12210118.0,2020-10-25 15:00:00,10.0,32.0,2020.0,4.186320e+01,-8.766303e+01,7.000000e+00,1.400000e+01,3.000000e+00
75%,13149307.25,2023-07-13 16:36:15,17.0,53.0,2023.0,4.190715e+01,-8.762745e+01,9.000000e+00,1.800000e+01,5.000000e+00
max,14161190.0,2025-12-31 23:58:00,31.0,77.0,2025.0,4.202267e+01,-8.752453e+01,1.200000e+01,2.300000e+01,6.000000e+00
std,1228494.808984,NaN,7.008925,21.506605,2.922181,1.176823e+00,2.715888e+00,3.363113e+00,6.738574e+00,1.997895e+00


In [143]:
df_map.describe()

,ID,District,Community Area,Year,Latitude,Longitude,Month,Hour,DayOfWeek
count,2.454715e+06,2.454715e+06,2.454715e+06,2.454715e+06,2.454715e+06,2.454715e+06,2.454715e+06,2.454715e+06,2.454715e+06
mean,1.219336e+07,1.124520e+01,3.662978e+01,2.020413e+03,4.184455e+01,-8.766961e+01,6.603928e+00,1.278931e+01,3.016013e+00
std,1.232686e+06,7.008365e+00,2.149978e+01,2.928794e+00,8.686531e-02,5.940131e-02,3.358722e+00,6.728999e+00,1.998696e+00
min,2.224500e+04,1.000000e+00,1.000000e+00,2.016000e+03,3.661945e+01,-9.168657e+01,1.000000e+00,0.000000e+00,0.000000e+00
25%,1.130108e+07,6.000000e+00,2.300000e+01,2.018000e+03,4.176926e+01,-8.771163e+01,4.000000e+00,8.000000e+00,1.000000e+00
50%,1.220234e+07,1.000000e+01,3.200000e+01,2.020000e+03,4.186320e+01,-8.766304e+01,7.000000e+00,1.400000e+01,3.000000e+00
75%,1.315014e+07,1.700000e+01,5.300000e+01,2.023000e+03,4.190715e+01,-8.762745e+01,9.000000e+00,1.800000e+01,5.000000e+00
max,1.415992e+07,3.100000e+01,7.700000e+01,2.025000e+03,4.202267e+01,-8.752453e+01,1.200000e+01,2.300000e+01,6.000000e+00


In [142]:
df.to_csv('/Users/fuyanting/Documents/capstone/data/df.csv', index=False)
df_map.to_csv('/Users/fuyanting/Documents/capstone/data/df_map.csv', index=False)